# Session 2, train the addition transformer

Session 1 built the ~10.7M skeleton and the dataset. Session 2 trains it for real.

Hardware here is CPU-only, and the ~10.7M model costs about 6 seconds per step at batch 64, so a real run on it would take hours. Session 2 trains two smaller siblings of the same architecture and compares them: a toy scaling data point. The ~10.7M run waits for a GPU.

One correction from the first attempt: the loss must predict the NEXT token. The first version compared logits[t] to ids[t], which scores copying the current token; the loss looked tiny (0.0065) while exact-match was 0/200. The targets here are shifted so logits[t] is scored against ids[t+1], and loss is only on answer tokens.

## Setup

Same 15-token alphabet and model as Session 1. Batches are drawn from a pre-generated pool of 200,000 random addition examples so the step cost is the model, not Python string formatting.

In [1]:
import json, math, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_num_threads(2)

CHARS = list("0123456789+=")
SPECIALS = ["<pad>", "<bos>", "<eos>"]
VOCAB = CHARS + SPECIALS
stoi = {c: i for i, c in enumerate(VOCAB)}
itos = {i: c for c, i in stoi.items()}
PAD, BOS, EOS = stoi["<pad>"], stoi["<bos>"], stoi["<eos>"]
MAX_LEN = 16
POOL = 200_000

In [2]:
def encode_full(a, b, c):
    s = f"{a}+{b}={c}"
    ids = [BOS] + [stoi[ch] for ch in s] + [EOS]
    return ids + [PAD] * (MAX_LEN - len(ids))

def build_pool(n):
    # ys[t] is the target for logits[t]: the NEXT token, ids[t+1].
    # Loss is only on answer tokens (positions >= ans_start in ids).
    xs = torch.empty(n, MAX_LEN, dtype=torch.long)
    ys = torch.empty(n, MAX_LEN, dtype=torch.long)
    for i in range(n):
        a = random.randint(0, 999)
        b = random.randint(0, 999)
        c = a + b
        ids = encode_full(a, b, c)
        xs[i] = torch.tensor(ids)
        full = f"{a}+{b}={c}"
        ans_start = 1 + full.index("=") + 1
        tgt = []
        for t in range(MAX_LEN):
            if t == MAX_LEN - 1:
                tgt.append(-100)
            elif t + 1 >= ans_start and ids[t + 1] != PAD:
                tgt.append(ids[t + 1])
            else:
                tgt.append(-100)
        ys[i] = torch.tensor(tgt)
    return xs, ys

pool_x, pool_y = build_pool(POOL)
print("pool built", tuple(pool_x.shape))

pool built (200000, 16)


## Model

Same decoder-only transformer as Session 1, with the loss shifted so logits[t] predicts token t+1.

In [3]:
class Block(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x, mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_ff, max_len=32):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1)
        for blk in self.blocks:
            x = blk(x, mask)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

@torch.no_grad()
def greedy_answer(model, a, b):
    model.eval()
    ids = [BOS] + [stoi[c] for c in f"{a}+{b}="]
    x = torch.tensor([ids])
    for _ in range(6):
        logits, _ = model(x)
        nxt = int(logits[0, -1].argmax())
        ids.append(nxt)
        x = torch.tensor([ids])
        if nxt == EOS:
            break
    out = "".join(itos[i] for i in ids if itos[i] not in ("<pad>", "<bos>", "<eos>"))
    return out.split("=")[1] if "=" in out else ""

@torch.no_grad()
def evaluate(model, n=200):
    correct = 0
    for _ in range(n):
        a = random.randint(0, 999); b = random.randint(0, 999); c = a + b
        if greedy_answer(model, a, b) == str(c):
            correct += 1
    return correct / n

## Train: small (603,008 params, d=128, 3 layers)

AdamW, lr 3e-4, cosine decay, batch 64, 400 steps. Loss is next-token cross-entropy on answer tokens only.

In [4]:
def train_run(name, d_model, n_layer, n_head, d_ff, steps, batch=64, lr=3e-4):
    model = TinyTransformer(len(VOCAB), d_model=d_model, n_layer=n_layer, n_head=n_head, d_ff=d_ff)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.1)
    print(f"=== {name}: d={d_model} L={n_layer} params={n_params:,} steps={steps} batch={batch} ===")
    t0 = time.time()
    for step in range(1, steps + 1):
        idx = torch.randint(0, pool_x.shape[0], (batch,))
        x, y = pool_x[idx], pool_y[idx]
        logits, loss = model(x, targets=y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        if step % 100 == 0 or step == 1:
            print(f"step {step:4d} loss {loss.item():.4f} ({time.time()-t0:.0f}s elapsed)")
    acc = evaluate(model, n=200)
    print(f"{name} final loss {loss.item():.4f}, exact-match accuracy on 200 held-out problems: {acc:.3f} ({int(acc*200)}/200)")
    return model, acc

small_model, small_acc = train_run("small", d_model=128, n_layer=3, n_head=4, d_ff=512, steps=400)

=== small: d=128 L=3 params=603,008 steps=400 batch=64 ===
step    1 loss 2.9554 (0s elapsed)
step  100 loss 1.5624 (11s elapsed)
step  200 loss 1.4312 (22s elapsed)
step  300 loss 1.3295 (34s elapsed)
step  400 loss 1.3444 (46s elapsed)
small final loss 1.3444, exact-match accuracy on 200 held-out problems: 0.010 (2/200)


## Train: medium (1,791,744 params, d=192, 4 layers)

Same recipe, same 400 steps, same batch. The comparison is the point: does the bigger model learn addition better at matched steps?

In [5]:
medium_model, medium_acc = train_run("medium", d_model=192, n_layer=4, n_head=4, d_ff=768, steps=400)

=== medium: d=192 L=4 params=1,791,744 steps=400 batch=64 ===
step    1 loss 2.8746 (0s elapsed)
step  100 loss 1.4611 (30s elapsed)
step  200 loss 1.2631 (58s elapsed)
step  300 loss 1.1725 (86s elapsed)
step  400 loss 1.1496 (114s elapsed)
medium final loss 1.1496, exact-match accuracy on 200 held-out problems: 0.010 (2/200)


## Result

| model | params | steps | final loss | exact-match (n=200) |
|---|---|---|---|---|
| small | 603,008 | 400 | 1.3444 | 0.010 (2/200) |
| medium | 1,791,744 | 400 | 1.1496 | 0.010 (2/200) |

The bigger model reaches lower loss (1.15 vs 1.34), the scaling direction is right, but neither learns 3-digit addition in 400 steps: exact-match is 1% for both. The loss is still falling at step 400, so the run is undertrained, not incapable. Addition is a hard task for a small transformer: it has to learn carries, and 400 steps at batch 64 is only 25,600 examples. The next step is a longer run, and then the ~10.7M model on a GPU.

Full write-up in ../writeups/session-02-writeup.md.